# Tools

In [5]:
# from openai.types.responses import response
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:openai/gpt-oss-20b")
response=model.invoke("why do the parrot talk")
response.content

'Parrots are famous for “talking”, but it’s not the same as human speech. They are highly sophisticated vocal learners that mimic sounds they hear in their environment. Here’s why they can “talk” and what makes it possible:\n\n| What they do | How it happens | Why it matters |\n|--------------|----------------|----------------|\n| **Mimicry** | Parrots hear a sound and repeat it. Their syrinx (the bird’s vocal organ) is exceptionally flexible, allowing them to produce a wide range of tones. | Mimicry helps them communicate within flocks, establish social bonds, and, in captivity, bond with humans. |\n| **Vocal learning** | Unlike most birds, parrots can learn new sounds after birth. They have a large portion of their brain dedicated to vocal learning—similar in function to the human Broca’s area. | This ability lets them adapt to new sounds and environments, which is why they can pick up words, music, and even other animal calls. |\n| **Social interaction** | Parrots live in complex so

In [6]:
from langchain.tools import tool

@tool
def get_weather(location:str)-> str:
    """ get the weather at a location"""
    return f" its the sunny in {location}"

model_with_tools=model.bind_tools([get_weather])

In [7]:
response=model_with_tools.invoke("what's the weather like in boston?")
print(response)
for tool_call in response.tool_calls:
    print(f"tools: { tool_call['name']}")
    print(f"Args: { tool_call['args']}")


content='' additional_kwargs={'reasoning_content': 'We need to use the get_weather function.', 'tool_calls': [{'id': 'fc_991e5154-41fc-4b3c-baa6-dee6f12941f9', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 129, 'total_tokens': 162, 'completion_time': 0.034301906, 'completion_tokens_details': {'reasoning_tokens': 10}, 'prompt_time': 0.008186037, 'prompt_tokens_details': None, 'queue_time': 0.249018095, 'total_time': 0.042487943}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_4f7e7dc26e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0a0bb-e234-75a0-9326-fe645b9d489f-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_991e5154-41fc-4b3c-baa6-dee6f12941f9', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 129, 'output_to

# Tool execution Loop

In [ ]:
# step 1  model generate the tools calls 
messages=[{"role":"user","content":"what's the weather in boston?"}]
ai_msg=model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: execute tools and collect result 
for tool_call in ai_msg.tool_calls:
    # excute the tool with the generated arguments
    tool_result=get_weather.invoke(tool_call)
    messages.append(tool_result)

# step 3
final_response=model_with_tools.invoke(messages)
print(final_response.text)


It’s sunny in Boston.


In [10]:
messages

[{'role': 'user', 'content': "what's the weather in boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the function get_weather with location "Boston".', 'tool_calls': [{'id': 'fc_99b38c6c-8317-4072-b7ce-27474b3854f6', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 128, 'total_tokens': 165, 'completion_time': 0.049912909, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.007946873, 'prompt_tokens_details': None, 'queue_time': 0.378008277, 'total_time': 0.057859782}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_d3e146e1a5', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0a0bb-e3b7-7aa2-8925-295f31098016-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_99b38c6c-8317-4072-b7ce-2747